# NLP Fine-Tuning Chatbot Project

In this notebook, you will:
1. Fine-tune a small language model on a dataset of your choice
2. Compare the model's responses **before** and **after** fine-tuning
3. Launch a Gradio chatbot you can share via a public link

**Just run each cell in order!** The default settings (TinyLlama + QA dataset + LoRA) work out of the box.

---
## Step 1: Setup

**Choose ONE of the two options below** depending on where you are running this notebook.

- **Option A:** Google Colab (recommended — free GPU)
- **Option B:** Local machine (your own computer)

### Option A: Google Colab Setup

Run the two cells below to clone the repo and install dependencies.

**Skip this section if you are running locally.**

In [ ]:
# ============================================
# COLAB ONLY — Clone the project from GitHub
# ============================================
!git clone https://github.com/ratulalahy/nlp_course_sample_chat_fine_tune.git
%cd nlp_course_sample_chat_fine_tune

In [ ]:
# ============================================
# COLAB ONLY — Install dependencies
# ============================================
# This installs all required packages (torch, transformers, peft, gradio, etc.)
# The -q flag keeps the output quiet so it's easier to read.
!pip install -r requirements.txt -q
print("All packages installed!")

### Option B: Local Setup

If you are running this notebook on your own machine, open a terminal and run:

```bash
# 1. Clone the repo
git clone https://github.com/ratulalahy/nlp_course_sample_chat_fine_tune.git
cd nlp_course_sample_chat_fine_tune

# 2. Create a virtual environment
python -m venv venv
source venv/bin/activate        # On Windows: venv\Scripts\activate

# 3. Install dependencies
pip install -r requirements.txt

# 4. Launch this notebook
jupyter notebook notebook.ipynb
```

Then continue from **Step 2** below.

**Note:** Training on CPU (no GPU) will be slow but works for small models like TinyLlama and Qwen3-0.6B.

---
## Step 2: Check GPU

Make sure you have a GPU available. On Colab, if this says `No GPU`, go to **Runtime > Change runtime type > T4 GPU**.

On a local machine without a GPU, training will still work — just slower.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected. Training will run on CPU (slower but works).")
    print("If you're on Colab: Runtime > Change runtime type > T4 GPU")
    print("If you're local: this is fine for small models like TinyLlama.")

---
## Step 3: Choose Your Configuration

Edit the settings below to pick your **model** and **dataset**.

The defaults (TinyLlama + QA Bot + LoRA) are a great starting point.

After your first run, come back here, change the settings, and re-run to compare!

In [ ]:
import yaml

# Load the default config
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# =============================================
# CHANGE THESE SETTINGS TO EXPERIMENT!
# =============================================

# --- Pick a model (uncomment ONE) ---
config["model"]["name"] = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"    # ~700MB, fastest
# config["model"]["name"] = "Qwen/Qwen3-0.6B"                     # ~400MB, ultra-light
# config["model"]["name"] = "HuggingFaceTB/SmolLM2-1.7B-Instruct" # 1.7B
# config["model"]["name"] = "google/gemma-4-e2b"                   # ~2B, newest
# config["model"]["name"] = "microsoft/Phi-3.5-mini-instruct"     # 3.8B

# --- Pick a dataset (uncomment ONE) ---
config["dataset"]["path"] = "datasets/qa_bot.json"         # General Q&A
# config["dataset"]["path"] = "datasets/uvu_bot.json"      # UVU FAQ
# config["dataset"]["path"] = "datasets/cs_assistant.json"  # CS concepts

# --- Pick training method ---
config["training"]["method"] = "lora"   # "lora" (recommended) or "full"

# --- Training settings (defaults work well) ---
config["training"]["epochs"] = 3
config["training"]["batch_size"] = 4
config["training"]["learning_rate"] = 2.0e-4

# Save updated config
with open("config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Model:   {config['model']['name']}")
print(f"Dataset: {config['dataset']['path']}")
print(f"Method:  {config['training']['method']}")
print(f"Epochs:  {config['training']['epochs']}")
print("\nConfig saved! Ready to fine-tune.")

---
## Step 4: Test Dataset Loading
Quick sanity check — make sure your dataset loads correctly before training.

In [ ]:
!python data_utils.py

---
## Step 5: Fine-Tune the Model

This is the main training step. It will:
1. Load the base model from HuggingFace
2. Run a **baseline test** (model responses BEFORE fine-tuning)
3. Train the model on your dataset
4. Run a **post-training test** (model responses AFTER fine-tuning)
5. Save the model to `output/` and the experiment log to `experiments/`

**Expected time:** ~2-5 minutes on a T4 GPU with TinyLlama.

In [ ]:
!python finetune.py

---
## Step 6: Launch the Chatbot

This starts a Gradio web interface with a **public shareable link**.

A URL like `https://xxxxx.gradio.live` will appear — click it to chat with your fine-tuned model!

**Portfolio tip:** Copy the public URL. Anyone can try your chatbot for 72 hours.

In [ ]:
!python app.py

---
## Step 7: Compare Experiments

After running multiple experiments (different models, datasets, or settings), run this cell to see all your results side by side.

**To run another experiment:** Go back to Step 3, change the settings, then re-run Steps 4-6. Each run is saved automatically.

In [ ]:
import json
import os

experiments_dir = "experiments"

if not os.path.exists(experiments_dir):
    print("No experiments yet! Run Step 5 (finetune.py) first.")
else:
    # Load all experiment files
    experiments = []
    for fname in sorted(os.listdir(experiments_dir)):
        if fname.endswith(".json"):
            with open(os.path.join(experiments_dir, fname)) as f:
                experiments.append(json.load(f))

    # --- Summary Table ---
    print(f"Found {len(experiments)} experiment(s):\n")
    print(f"{'#':<4} {'Model':<35} {'Dataset':<20} {'Method':<8} {'Loss':<8} {'Time':<8}")
    print("-" * 85)

    for i, exp in enumerate(experiments, 1):
        c = exp["config"]
        r = exp["results"]
        model_short = c["model_name"].split("/")[-1][:33]
        dataset_short = os.path.basename(c["dataset"]).replace(".json", "")
        time_str = f"{r['training_time_seconds']:.0f}s"
        print(f"{i:<4} {model_short:<35} {dataset_short:<20} {c['method']:<8} {r['final_loss']:<8} {time_str:<8}")

    # --- Before vs After for the latest experiment ---
    latest = experiments[-1]
    print(f"\n{'=' * 85}")
    print(f"Latest experiment — Before vs After comparison:")
    print(f"{'=' * 85}")

    for b, a in zip(latest["baseline_responses"], latest["finetuned_responses"]):
        print(f"\nQ: {b['question']}")
        print(f"  BEFORE: {b['response'][:150]}")
        print(f"  AFTER:  {a['response'][:150]}")

---
## What's Next?

Try these experiments and compare results:

1. **Change the dataset** — Go back to Step 3, switch to `uvu_bot.json` or `cs_assistant.json`
2. **Change the model** — Try `Qwen/Qwen3-0.6B` or `google/gemma-4-e2b`
3. **Change training settings** — More epochs? Higher learning rate?
4. **Use your own data** — Create a custom JSON dataset (see `examples/load_hf_dataset.py`)
5. **Compare LoRA vs Full** — Switch `method` to `"full"` (warning: needs more GPU memory)

Each run saves to `experiments/` — use Step 7 to compare them all!